# KLUE-NER 1차 학습 (L4) — Colab

`klue/bert-base`는 NER 헤드가 없는 MLM 인코더다. KLUE-NER 공개 데이터로 헤드를 학습시킨다.

## 실행 전 설정

**런타임 → 런타임 유형 변경 → 하드웨어 가속기: `T4 GPU`**

## 이 노트북이 전제하는 것

- **`/content/`는 세션이 끝나면 사라진다.** 산출물은 전부 Drive에 쓴다.
- 세션이 끊길 수 있다. 두 단계로 대비한다.
  - **run 단위** — 끝난 run은 `metrics.json`을 보고 건너뛴다
  - **스텝 단위** — `SAVE_STEPS`마다 체크포인트를 남기고 거기서 이어서 돌린다

  끊기면 **셀을 다시 실행하기만 하면** 된다. 처음부터 다시 돌지 않는다.

예상 소요: T4 기준 4개 run 전부 **약 40~60분**. 무료 Colab은 브라우저를 닫으면
세션이 끊기니 그동안 탭을 열어둔다.

## 1. GPU 확인

GPU가 없으면 여기서 멈춘다. CPU로는 4개 run에 19시간이 걸린다(실측).

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA 사용가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

assert torch.cuda.is_available(), (
    "GPU가 꺼져 있다. 런타임 > 런타임 유형 변경 > 하드웨어 가속기를 T4 GPU로 바꿀 것"
)

## 2. Drive 마운트

**학습을 시작하기 전에 반드시 한다.** `/content/`에 저장했다가 나중에 옮기려 하면
그 '나중'이 오기 전에 런타임이 회수된다.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

OUTPUT_DIR = Path("/content/drive/MyDrive/privacy-proxy/l4-ner")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 마운트가 실제로 됐는지 쓰기로 확인한다. mount()가 조용히 실패하는 경우가 있다.
probe = OUTPUT_DIR / ".write_test"
probe.write_text("ok")
assert probe.read_text() == "ok", "Drive에 쓸 수 없다. 마운트를 다시 확인할 것"
probe.unlink()

print("산출물 경로:", OUTPUT_DIR)
print("기존 내용:", sorted(p.name for p in OUTPUT_DIR.iterdir()) or "(비어 있음)")

## 3. 코드 가져오기

**권장: Drive에 `l4/` 를 두기.** 세션이 끊겨도 다시 올릴 필요가 없다.

로컬에서 `l4/` 폴더를 Google Drive의
`MyDrive/privacy-proxy/code/l4` 에 복사해두면 아래 셀이 알아서 찾는다.

없으면 `l4_ner.zip` 업로드 창이 뜬다. 로컬 저장소 루트에서 만든다:

```bash
zip -r l4_ner.zip l4 -x "l4/kaggle/*" -x "l4/colab/*" -x "*__pycache__*"
```

> private 저장소라 `git clone` 에는 GitHub 토큰이 필요하다. 개인정보 취급
> 프로젝트에서 외부로 토큰을 넘기지 않으려고 파일 전달을 기본 경로로 둔다.

In [ ]:
import shutil
import sys
import zipfile

CODE = Path("/content/code")
CODE.mkdir(parents=True, exist_ok=True)

DRIVE_CODE = Path("/content/drive/MyDrive/privacy-proxy/code")


def locate(root: Path) -> Path | None:
    """ner/bio.py 를 기준점으로 삼아 그 상위를 패키지로 본다.

    l4/ 로 감싼 구조든 ner/ 가 바로 있는 구조든 똑같이 처리된다.
    """
    if not root.exists():
        return None
    for marker in sorted(root.rglob("ner/bio.py")):
        target = CODE / "l4"
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(marker.parent.parent, target)
        return target
    return None


l4_dir = locate(DRIVE_CODE)

if l4_dir is None:
    print(f"{DRIVE_CODE} 에 코드가 없다. zip을 업로드한다.")
    from google.colab import files

    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            with zipfile.ZipFile(name) as archive:
                archive.extractall(CODE / "_unzipped")
    l4_dir = locate(CODE / "_unzipped")

    # 다음 세션에서 다시 올리지 않도록 Drive에 넣어둔다
    if l4_dir is not None:
        DRIVE_CODE.mkdir(parents=True, exist_ok=True)
        backup = DRIVE_CODE / "l4"
        if backup.exists():
            shutil.rmtree(backup)
        shutil.copytree(l4_dir, backup)
        print(f"다음 세션을 위해 {backup} 에 복사해뒀다.")

if l4_dir is None:
    raise SystemExit(
        "l4 패키지를 찾지 못했다.\n"
        f"  1) {DRIVE_CODE}/l4 에 코드를 두거나\n"
        "  2) 업로드 창에 l4_ner.zip 을 올릴 것"
    )

sys.path.insert(0, str(l4_dir.parent))
print("코드 경로:", l4_dir)
print("모듈:", sorted(p.name for p in (l4_dir / "ner").glob("*.py")))

import l4.ner.bio  # noqa: F401  임포트가 되는지 여기서 확인한다

print("임포트 정상")

## 4. 의존성

Colab에 transformers/torch는 이미 있다. 버전이 로컬과 달라도
`build_training_arguments()` 가 시그니처를 보고 맞춰준다.

In [ ]:
!pip install -q seqeval

import datasets
import transformers

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

## 5. 파이프라인 자체 검증

학습 전에 **평가 경로가 정보를 잃지 않는지** 확인한다.
정답을 그대로 통과시켰을 때 F1이 0.98 근처여야 한다.

0.73 근처가 나오면 엔티티 내부 공백 잇기가 깨진 것이다. KLUE-NER 정답
엔티티의 14%가 공백을 품는데(`'서울 강남구'`), 토크나이저는 공백을 어느
토큰에도 넣지 않는다. 이걸 안 이으면 엔티티가 두 조각으로 갈라져서,
**모델이 멀쩡해도 F1 0.73이 천장**이 된다.

In [ ]:
from datasets import load_dataset
from l4.ner.bio import merge_bio_spans, spans_to_bio
from l4.ner.data import encode_example, expand_subword_to_char, restore_text
from l4.ner.labels import load_label_set
from l4.ner.metrics import canonicalize_gold, score
from transformers import AutoTokenizer

ds = load_dataset("klue/klue", "ner")
label_set = load_label_set(ds)
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

print("라벨:", label_set.names)
print("O 인덱스:", label_set.outside_id, "(12여야 정상)")

gold_seqs, pred_seqs = [], []
for example in ds["validation"].select(range(1000)):
    text = restore_text(example["tokens"])
    encoded = encode_example(example, tokenizer, 512)
    ids = [label_set.outside_id if v == -100 else v for v in encoded["labels"]]
    tags = expand_subword_to_char(encoded["offset_mapping"], ids, label_set, len(text))
    pred_seqs.append(spans_to_bio(merge_bio_spans(tags, text), len(text)))
    gold_seqs.append(canonicalize_gold(example["ner_tags"], label_set, text))

ceiling = score(gold_seqs, pred_seqs)
print(f"\n평가 경로 상한: entity F1={ceiling['f1']:.4f}  PS recall={ceiling['ps_recall']:.4f}")
assert ceiling["f1"] > 0.95, "평가 경로가 정보를 잃고 있다. 공백 잇기를 확인할 것"
print("정상. 학습으로 넘어간다.")

## 6. 학습

### 세션이 끊기면

**이 셀을 다시 실행하면 된다.** 끝난 run은 건너뛰고, 하던 run은 마지막
체크포인트에서 이어간다.

### `SAVE_STEPS` 를 정할 때

체크포인트 하나가 **1.3GB**다(모델 420MB + 옵티마이저 상태 840MB).
Drive 쓰기는 느려서 너무 자주 저장하면 학습보다 저장이 오래 걸린다.

| 값 | 효과 |
| --- | --- |
| `0` | 저장 안 함. 가장 빠르지만 끊기면 그 run은 처음부터 |
| `500` | 기본값. T4에서 epoch당 2~3회 저장 |
| `200` | 더 촘촘하지만 Drive I/O가 학습 시간을 넘길 수 있다 |

run 하나가 15분쯤이라 사실상 run 단위 건너뛰기가 주된 방어선이고,
스텝 체크포인트는 보조 수단이다.

In [ ]:
SAVE_STEPS = 500
SAVE_TOTAL_LIMIT = 1  # Drive 용량을 위해 최신 것 하나만 남긴다

from l4.ner.experiments import EXPERIMENTS, run

for experiment in EXPERIMENTS:
    print(
        f"{experiment.run_name:22s} lr={experiment.lr:<7} bs={experiment.batch_size} "
        f"ep={experiment.epochs} len={experiment.max_length}  # {experiment.note}"
    )
print()

results = run(
    output_root=OUTPUT_DIR,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    skip_completed=True,
)

## 7. 결과

In [ ]:
import pandas as pd

summary = pd.read_csv(OUTPUT_DIR / "summary.csv")
display(summary)

print("\n주 지표는 전체 F1이지만, L4가 실제로 쓰는 태그는 PS(인명)·LC(지명)뿐이다.")
print("PS recall = 인명 누락률. 누락은 곧 개인정보 유출이라 따로 본다.")
print(f"평가 경로 상한이 {ceiling['f1']:.4f} 이므로 그 대비로 읽을 것.\n")
for _, row in summary.iterrows():
    print(
        f"{row['run_name']:22s} F1={row['f1']:.4f}  PS_recall={row['PS_recall']:.4f}  "
        f"지연 p50={row['latency_p50_s'] * 1000:.1f}ms"
    )

In [ ]:
from IPython.display import Image
from IPython.display import display as show
from l4.ner.visualize import render

render(OUTPUT_DIR)
for name in ["f1_curves.png", "per_tag.png", "latency_tradeoff.png"]:
    path = OUTPUT_DIR / "figures" / name
    if path.exists():
        show(Image(str(path)))

## 8. 조사 경계 확인

모델이 `'김철수에게'` 를 통째로 잡는지, `'김철수'` 만 잡는지 본다.

**이 출력이 다음 주 `alignment/`(조사 경계 중재) 모듈의 설계 입력이다.**

In [ ]:
from l4.ner.check_josa import report

report(OUTPUT_DIR / "klue-ner-base-e3" / "model", max_length=512, include_ecommerce=True)

## 8-1. 조사 중재 정량 평가

위 셀은 9문장만 눈으로 본 것이다. 여기서는 validation 전체에 대해
`alignment.py` 적용 전/후 PS recall·F1을 잰다. 모델을 다시 학습하지
않는다 — 이미 끝난 `klue-ner-base-e3`의 로짓을 한 번만 뽑아서 두 번 채점한다.

In [ ]:
import json

from l4.ner.evaluate_alignment import run as evaluate_alignment

alignment_result = evaluate_alignment(OUTPUT_DIR / "klue-ner-base-e3" / "model")

alignment_path = OUTPUT_DIR / "klue-ner-base-e3" / "alignment_eval.json"
alignment_path.write_text(
    json.dumps(alignment_result, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"\n저장: {alignment_path}")

## 9. 정리 및 다음 단계

체크포인트(각 1.3GB)를 지우고, 저장소에 커밋할 것만 따로 모은다.

**가중치는 깃에 올리지 않는다.** 430MB라 GitHub 100MB 제한에 걸리고,
`.gitignore`에도 막혀 있다. 깃에는 **지표와 그림만** 올린다.

In [ ]:
import shutil

# 체크포인트는 재개용이라 학습이 끝나면 필요 없다. 하나에 1.3GB다.
for checkpoints in OUTPUT_DIR.glob("*/checkpoints"):
    shutil.rmtree(checkpoints, ignore_errors=True)
    print("삭제:", checkpoints)

# 기준선 외의 가중치도 지운다. 비교는 summary.csv로 충분하다.
for model_dir in OUTPUT_DIR.glob("*/model"):
    if model_dir.parent.name != "klue-ner-base-e3":
        shutil.rmtree(model_dir, ignore_errors=True)
        print("삭제:", model_dir)

# 저장소에 커밋할 것만 모은다
commit_dir = OUTPUT_DIR / "for_git"
shutil.rmtree(commit_dir, ignore_errors=True)
commit_dir.mkdir(parents=True)

shutil.copy(OUTPUT_DIR / "summary.csv", commit_dir)
shutil.copytree(OUTPUT_DIR / "figures", commit_dir / "figures", dirs_exist_ok=True)
for run_dir in OUTPUT_DIR.glob("klue-ner-*"):
    target = commit_dir / run_dir.name
    target.mkdir(exist_ok=True)
    for name in ("metrics.json", "log_history.csv", "alignment_eval.json"):
        source = run_dir / name
        if source.exists():
            shutil.copy(source, target)

size = sum(f.stat().st_size for f in commit_dir.rglob("*") if f.is_file())
print(f"\n커밋용 묶음: {commit_dir}  ({size / 1e6:.1f} MB)")
for path in sorted(commit_dir.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(commit_dir))

model_dir = OUTPUT_DIR / "klue-ner-base-e3" / "model"
model_size = sum(f.stat().st_size for f in model_dir.rglob("*") if f.is_file())
print(f"\n기준선 가중치: {model_dir}  ({model_size / 1e6:.0f} MB) — 깃에 올리지 않는다")

### 학습 이후 할 일

1. **지표·그림 커밋** — Drive의 `for_git/` 을 로컬로 가져와서

   ```bash
   cp -r ~/Drive/privacy-proxy/l4-ner/for_git docs/experiments/klue-ner-1st
   git add docs/experiments/klue-ner-1st && git commit -m "[detection-ecommerce] KLUE-NER 1차 학습 결과 기록"
   ```

2. **가중치는 Drive에 그대로 둔다** — 팀원은 Drive 공유 폴더에서 받는다.
   버전 관리가 안 되니 폴더명으로 구분한다(`klue-ner-v1`, `v2`).
   HF Hub private을 쓸 거면 `huggingface-cli upload ... --private`.

3. **ADR 작성** — `docs/decisions/0002-klue-ner-1차-학습-결과.md`
   어떤 설정을 골랐고 왜인지, F1이 상한 대비 어디쯤인지.

4. **`config.py` 에 모델 경로 항목 추가** — 파이프라인이 모델을 찾을 수 있게.

> **P3에서는 규칙이 달라진다.** 이번 모델은 공개 데이터로만 학습해서 공유해도
> 되지만, 이커머스 CS 실데이터로 학습한 2차 모델은 **가중치 자체가 개인정보**다.
> 신경망은 학습 데이터를 암기하고 추출 공격으로 되뽑힐 수 있다. 외부 서비스에
> 올리면 안 되고 사내 저장소나 S3(KMS 암호화)에 둬야 한다. ADR로 미리 정해둘 것.